In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("used_cars.csv")

In [3]:
df

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"
...,...,...,...,...,...,...,...,...,...,...,...,...
4004,Bentley,Continental GT Speed,2023,714 mi.,Gasoline,6.0L W12 48V PDI DOHC Twin Turbo,8-Speed Automatic with Auto-Shift,C / C,Hotspur,None reported,Yes,"$349,950"
4005,Audi,S4 3.0T Premium Plus,2022,"10,900 mi.",Gasoline,349.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,Black,Black,None reported,Yes,"$53,900"
4006,Porsche,Taycan,2022,"2,116 mi.",NaN,Electric,Automatic,Black,Black,None reported,NaN,"$90,998"
4007,Ford,F-150 Raptor,2020,"33,000 mi.",Gasoline,450.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,A/T,Blue,Black,None reported,Yes,"$62,999"


In [4]:
import re
def remove_mi(text):
    pattern="[a-z\.\,]"
    c=re.sub(pattern,"",text)
    return c

df['milage']=df['milage'].apply(remove_mi).astype('int64')

<>:3: SyntaxWarning: invalid escape sequence '\.'
<>:3: SyntaxWarning: invalid escape sequence '\.'
C:\Users\abhin\AppData\Local\Temp\ipykernel_22196\2963429368.py:3: SyntaxWarning: invalid escape sequence '\.'
  pattern="[a-z\.\,]"


In [5]:
def convert_fuel(fuel):
    if fuel == 'Plug-In Hybrid':
        return 'Hybrid'
    elif fuel in ['not supported','-','–']:
        return "Other"
    else:
        return fuel
    
df['fuel_type']=df['fuel_type'].apply(convert_fuel)

In [6]:
df['fuel_type']=df['fuel_type'].replace(np.nan,'Electric')

In [7]:
df['engine_hp'] = df['engine'].str.extract(r'(\d+\.?\d*)\s?HP', expand=False).astype(float)

In [8]:
df['liters'] = df['engine'].str.extract(r'(\d+\.?\d*)\s?L', expand=False).astype(float)

In [9]:
#Fill missing HP based on the median HP for that specific engine size
df['engine_hp'] = df['engine_hp'].fillna(df.groupby('liters')['engine_hp'].transform('median'))

In [10]:
df['engine_hp'] = df['engine_hp'].fillna(df['engine_hp'].median())

In [11]:
def clean_transmission(text):
    word=str(text).lower()
    manual=['manual','m/t','mt']
    cvt=['cvt','variable']
    single_speed=['single-speed','1-speed','fixed gear']
    auto=['automatic','a/t','at','auto']
    
    if any(m in word for m in manual):
        return "Manual"
    elif any(c in word for c in cvt):
        return "CVT"
    elif any(s in word for s in single_speed):
        return "Single-Speed"
    elif any(a in word for a in auto):
        return "Automatic"
    else:
        return "Other"

df['transmission_type']=df['transmission'].apply(clean_transmission)

In [12]:
df['accident']=df['accident'].replace({'At least 1 accident or damage reported':1, 'None reported':0})

C:\Users\abhin\AppData\Local\Temp\ipykernel_22196\1894609001.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['accident']=df['accident'].replace({'At least 1 accident or damage reported':1, 'None reported':0})


In [13]:
df['accident']=df['accident'].fillna(0).astype('int64')

In [14]:
df['clean_title']=df['clean_title'].replace('Yes',1)

C:\Users\abhin\AppData\Local\Temp\ipykernel_22196\185220089.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['clean_title']=df['clean_title'].replace('Yes',1)


In [15]:
df['clean_title']=df['clean_title'].fillna(0).astype('int64')

In [16]:
df['price']=df['price'].replace('[$,]',"",regex=True)

In [17]:
df['price']=df['price'].astype('int64')

In [41]:
df['engine displacement'] = df['engine'].str.extract(r'(\d+\.\d+)\s*L')

In [42]:
df['engine displacement'] = df['engine displacement'].fillna(df['engine'].str.extract(r'(\d+\.\d+)\s*LITER')[0])

In [44]:
df['is_v_engine'] = df['engine'].str.contains(r'V\d+', case=False, na=False)

In [51]:
def v_engine(v):
    if v:
        return 1
    else:
        return 0
df['is_v_engine']=df['is_v_engine'].apply(v_engine)

In [52]:
df

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,...,liters,transmission_type,log_price,log_milage,log_hp,log_year,engine displacement,is_v_engine,Vehicle_Age,Mileage_per_Year
0,Ford,Utility Police Interceptor Base,2013,51000,E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,1,...,3.7,Automatic,9.239899,10.839581,5.703782,7.607381,3.7,1,11,4636.363636
1,Hyundai,Palisade SEL,2021,34742,Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,1,...,3.8,Automatic,10.545473,10.455705,5.840642,7.611348,3.8,1,3,11580.666667
2,Lexus,RX 350 RX 350,2022,22372,Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,0,...,3.5,Automatic,10.907753,10.015565,5.686975,7.611842,3.5,0,2,11186.000000
3,INFINITI,Q50 Hybrid Sport,2015,88900,Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,0,...,3.5,Automatic,9.648595,11.395267,5.869297,7.608374,3.5,1,9,9877.777778
4,Audi,Q3 45 S line Premium Plus,2021,9835,Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,0,...,2.0,Automatic,10.463075,9.193703,5.480639,7.611348,2.0,0,3,3278.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4003,Mercedes-Benz,E-Class E 300 4MATIC,2018,53705,Gasoline,241.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,A/T,Black,Black,1,...,2.0,Automatic,10.161998,10.891261,5.484797,7.609862,2.0,0,6,8950.833333
4005,Audi,S4 3.0T Premium Plus,2022,10900,Gasoline,349.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,Black,Black,0,...,3.0,Other,10.894886,9.296518,5.855072,7.611842,3.0,1,2,5450.000000
4006,Porsche,Taycan,2022,2116,Electric,Electric,Automatic,Black,Black,0,...,NaN,Automatic,11.418593,7.657283,5.730100,7.611842,NaN,0,2,1058.000000
4007,Ford,F-150 Raptor,2020,33000,Gasoline,450.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,A/T,Blue,Black,0,...,3.5,Automatic,11.050874,10.404263,6.109248,7.610853,3.5,1,4,8250.000000


In [46]:
df['Vehicle_Age'] = 2024 - df['model_year']

In [47]:
df['Mileage_per_Year'] = df.apply(
    lambda row: row['milage'] / row['Vehicle_Age'] if row['Vehicle_Age'] > 0 else row['milage'],
    axis=1
)

In [20]:
columns = ['engine_hp', 'price', 'milage']
for col in columns:
    Q1 = df[col].quantile(0.25)  
    Q3 = df[col].quantile(0.75)   
    IQR = Q3 - Q1                    
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

In [21]:
df.shape

(3635, 15)

In [22]:
df['log_price']=np.log(df['price'])

In [23]:
df['log_milage']=np.log(df['milage'])

In [24]:
df['log_hp']=np.log(df['engine_hp'])

In [25]:
df['log_year']=np.log(df['model_year'])

In [54]:
df['log_milage_per_year']=np.log(df['Mileage_per_Year'])

In [55]:
df

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,...,transmission_type,log_price,log_milage,log_hp,log_year,engine displacement,is_v_engine,Vehicle_Age,Mileage_per_Year,log_milage_per_year
0,Ford,Utility Police Interceptor Base,2013,51000,E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,1,...,Automatic,9.239899,10.839581,5.703782,7.607381,3.7,1,11,4636.363636,8.441686
1,Hyundai,Palisade SEL,2021,34742,Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,1,...,Automatic,10.545473,10.455705,5.840642,7.611348,3.8,1,3,11580.666667,9.357092
2,Lexus,RX 350 RX 350,2022,22372,Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,0,...,Automatic,10.907753,10.015565,5.686975,7.611842,3.5,0,2,11186.000000,9.322418
3,INFINITI,Q50 Hybrid Sport,2015,88900,Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,0,...,Automatic,9.648595,11.395267,5.869297,7.608374,3.5,1,9,9877.777778,9.198043
4,Audi,Q3 45 S line Premium Plus,2021,9835,Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,0,...,Automatic,10.463075,9.193703,5.480639,7.611348,2.0,0,3,3278.333333,8.095090
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4003,Mercedes-Benz,E-Class E 300 4MATIC,2018,53705,Gasoline,241.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,A/T,Black,Black,1,...,Automatic,10.161998,10.891261,5.484797,7.609862,2.0,0,6,8950.833333,9.099502
4005,Audi,S4 3.0T Premium Plus,2022,10900,Gasoline,349.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,Black,Black,0,...,Other,10.894886,9.296518,5.855072,7.611842,3.0,1,2,5450.000000,8.603371
4006,Porsche,Taycan,2022,2116,Electric,Electric,Automatic,Black,Black,0,...,Automatic,11.418593,7.657283,5.730100,7.611842,NaN,0,2,1058.000000,6.964136
4007,Ford,F-150 Raptor,2020,33000,Gasoline,450.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,A/T,Blue,Black,0,...,Automatic,11.050874,10.404263,6.109248,7.610853,3.5,1,4,8250.000000,9.017968


In [56]:
categorical_cols = [
    'brand','fuel_type','transmission_type'
]

In [57]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

In [58]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5
))
])

In [59]:
x=df[[
 'brand',
 'log_year',
 'log_milage',
 'fuel_type',
 'accident',
 'clean_title',
 'log_hp',
 'transmission_type',
 'is_v_engine',
 'Vehicle_Age',
 'log_milage_per_year'
]]
y=df['log_price']

In [66]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [67]:
pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)

In [68]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(np.exp(y_test), np.exp(y_pred))

In [70]:
print(mae)

5737.209748008717


In [64]:
df['price'].mean()

32984.95653370014

In [71]:
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=7,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8

))
])

In [72]:
pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)

In [73]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(np.exp(y_test), np.exp(y_pred))

In [74]:
print(mae)

5104.9129764604095


In [75]:
from sklearn.svm import SVR

pipeline_svr = Pipeline([
    ('preprocessing', preprocessor),
    ('model', SVR(
        kernel='rbf',  
        C=100.0,        
        epsilon=0.1,     
        gamma='scale'    
    ))
])

In [76]:
pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)

In [77]:
mae = mean_absolute_error(np.exp(y_test), np.exp(y_pred))

In [78]:
print(mae)

5104.9129764604095
